[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/46.%20The%20Competitive%20Port%20Pricing%20Problem/P46-Tier-2_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 46. The Competitive Port Pricing Problem

## Tier 2: Iterative Best Response Algorithm

### Goal

Implement an iterative best response algorithm for a competitive port-pricing game. Each port adjusts its prices in response to the prices currently set by other ports. The algorithm uses a logit choice model to allocate each shipping line's demand and seeks a stable set of prices under the model's price bounds and port-capacity limits.

The notebook takes port costs and capacities, shipping-line demands, and market parameters as inputs. It reports the sequence of prices and profits, whether the specified price-change stopping criterion was met, and diagnostic summaries and plots. It does not solve a system-wide profit-maximization problem.

### Modeling assumptions and scope

- Shipping lines choose among the modeled ports; there is no outside option, so each line's full demand is allocated among those ports.
- A shipping line's demand is allocated independently of other shipping lines. Port capacity, in contrast, is shared across all lines served by that port.
- A port's profit is revenue minus the specified per-line costs. Fixed costs, investment decisions, congestion effects, and price responses by shipping lines over time are outside this model.
- Prices are restricted to the specified interval. Each one-price best response is searched on a finite grid that also includes the current price. Consequently, the computed response is an approximate, grid-based best response, not necessarily the exact continuous-price optimum.
- Capacity is enforced when testing candidate price changes. The initial price matrix must also be feasible; if a randomly generated initial matrix is infeasible, the algorithm starts from the feasible upper-bound price matrix instead.
- Reaching the price-change tolerance is evidence of numerical stability for this update procedure. It does not by itself prove existence or uniqueness of an exact Nash equilibrium, nor does it guarantee global optimality.

### Choice model and profit

For shipping line \(j\), port \(i\)'s allocated volume is its demand multiplied by its logit choice probability:

$$
q_{ij}(p) = D_j \,\frac{\exp(-\beta p_{ij})}{\sum_{k=1}^{P}\exp(-\beta p_{kj})}.
$$

Here, \(P\) is the number of ports, \(D_j\) is the demand of shipping line \(j\), \(p_{ij}\) is port \(i\)'s price to that line, and \(\beta\) is the price-sensitivity parameter. A lower price increases a port's choice probability, while the denominator ensures that probabilities across the modeled ports sum to one. In the code, `calculate_market_share` returns \(q_{ij}\), measured in TEU, rather than a fraction.

Port \(i\)'s total profit and shared-capacity constraint are

$$
\Pi_i(p) = \sum_{j=1}^{L}(p_{ij}-c_{ij})q_{ij}(p),
\qquad
\sum_{j=1}^{L}q_{ij}(p) \leq K_i,
\qquad
p_{\min} \leq p_{ij} \leq p_{\max}.
$$

Here, \(L\) is the number of shipping lines, \(c_{ij}\) is port \(i\)'s cost for line \(j\), and \(K_i\) is its capacity. The shared capacity constraint matters because raising or lowering one price can change the total volume assigned to that port across all lines. The code checks this constraint for every candidate price using `check_capacity_constraints`.

### Sequential best response and stopping rule

The algorithm visits ports and shipping lines in order. At each visit it changes one price, holding the other current prices fixed, and selects the grid candidate that maximizes that port's profit contribution from the relevant line while keeping the port's total allocated volume within capacity. Because demand allocation is independent across lines, changing one line's price affects that line's profit contribution directly; the capacity constraint still couples the candidate's feasibility to all lines.

Writing \(p^{(t,r)}\) for the price matrix after update \(r\) of iteration \(t\), a coordinate update for port \(i\) and line \(j\) is

$$
p_{ij}^{(t,r+1)} \in
\underset{x\in\mathcal{G}_{ij}^{(t,r)}}{\operatorname{argmax}}
\;(x-c_{ij})q_{ij}\!\left(p^{(t,r)}_{-ij},x\right)
\quad\text{subject to}\quad
\sum_{\ell=1}^{L}q_{i\ell}\!\left(p^{(t,r)}_{-ij},x\right)\leq K_i.
$$

The notation \(p_{-ij}\) means all current prices except \(p_{ij}\); \(\mathcal{G}_{ij}^{(t,r)}\) is the finite price grid, including the current price; and \(x\) is a candidate price. The code implements this update in `best_response_price`, called sequentially by `iterate_best_response`.

After a complete pass through all port-line pairs, the stopping criterion is

$$
\Delta^{(t)} = \max_{i,j}\left|p_{ij}^{(t)}-p_{ij}^{(t-1)}\right| < \varepsilon,
$$

where \(\varepsilon\) is the convergence tolerance. This checks price stability between full iterations. It is not an independent proof that no port can improve its profit by a deviation outside the searched grid. The algorithm also stops at its iteration limit if the tolerance is not met.

### Computational considerations

The stored price history contains one \(P\times L\) matrix per completed iteration, so its storage grows as \(O(TPL)\) for \(T\) iterations. A full iteration makes \(PL\) coordinate updates. Each update tests a grid of \(G\) candidate prices; evaluating market shares and shared capacity across ports and lines gives a straightforward implementation cost on the order of \(O(GP^2L^2)\) per full iteration. Actual run time also depends on the number of iterations and the execution environment.

### Solution sequence

1. Define ports, shipping lines, and market parameters.
2. Initialize prices and check that they satisfy price bounds and capacity limits.
3. Sequentially update each port-line price by a capacity-feasible grid best response.
4. Record prices and profits after each full iteration and test the maximum price change against the tolerance.
5. Examine convergence behavior, initialization sensitivity, tolerance sensitivity, and performance on several problem sizes.

The example data below define the problem solved in this notebook. Results should be interpreted as outcomes of this specified model and algorithm, not as a general guarantee about competitive port markets.

In [ ]:
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class Port:
    """A port with a shared capacity and line-specific costs."""

    id: int
    name: str
    capacity: float
    costs: List[float]


@dataclass
class ShippingLine:
    """A shipping line with a specified total demand."""

    id: int
    name: str
    demand: float


@dataclass
class MarketParameters:
    """Market-wide price response and feasible price bounds."""

    price_sensitivity: float
    min_price: float
    max_price: float


class IterativeBestResponse:
    """Sequential, grid-based best responses for competitive port pricing."""

    def __init__(
        self,
        ports: List[Port],
        shipping_lines: List[ShippingLine],
        market_params: MarketParameters,
    ):
        if not ports:
            raise ValueError("At least one port is required.")
        if not shipping_lines:
            raise ValueError("At least one shipping line is required.")
        if any(port.capacity < 0 for port in ports):
            raise ValueError("Port capacities must be nonnegative.")
        if any(line.demand < 0 for line in shipping_lines):
            raise ValueError("Shipping-line demands must be nonnegative.")
        if market_params.price_sensitivity < 0:
            raise ValueError("Price sensitivity must be nonnegative.")
        if market_params.min_price > market_params.max_price:
            raise ValueError("The minimum price must not exceed the maximum price.")
        if any(len(port.costs) != len(shipping_lines) for port in ports):
            raise ValueError("Each port must have one cost for every shipping line.")

        self.ports = ports
        self.shipping_lines = shipping_lines
        self.market_params = market_params
        self.num_ports = len(ports)
        self.num_shipping_lines = len(shipping_lines)

        self.price_history = []
        self.profit_history = []

    def _validate_prices(self, prices: np.ndarray) -> None:
        """Check matrix shape and price bounds before evaluating the model."""
        expected_shape = (self.num_ports, self.num_shipping_lines)
        if prices.shape != expected_shape:
            raise ValueError(f"Prices must have shape {expected_shape}.")
        if not np.all(np.isfinite(prices)):
            raise ValueError("All prices must be finite.")
        if np.any(prices < self.market_params.min_price) or np.any(
            prices > self.market_params.max_price
        ):
            raise ValueError("Prices must lie within the market price bounds.")

    def calculate_market_share(
        self,
        prices: np.ndarray,
        port_idx: int,
        shipping_line_idx: int,
    ) -> float:
        """Return the demand volume allocated to one port-line pair, in TEU."""
        self._validate_prices(prices)
        beta = self.market_params.price_sensitivity
        demand = self.shipping_lines[shipping_line_idx].demand
        utilities = np.exp(-beta * prices[:, shipping_line_idx])
        total_utility = np.sum(utilities)
        if total_utility == 0.0:
            return 0.0
        return float(demand * utilities[port_idx] / total_utility)

    def calculate_port_profit(self, prices: np.ndarray, port_idx: int) -> float:
        """Compute a port's total variable profit across shipping lines."""
        self._validate_prices(prices)
        total_profit = 0.0
        for sl_idx in range(self.num_shipping_lines):
            price = prices[port_idx, sl_idx]
            cost = self.ports[port_idx].costs[sl_idx]
            volume = self.calculate_market_share(prices, port_idx, sl_idx)
            total_profit += (price - cost) * volume
        return total_profit

    def calculate_all_profits(self, prices: np.ndarray) -> np.ndarray:
        """Compute one total-profit value for each port."""
        return np.array(
            [self.calculate_port_profit(prices, port_idx)
             for port_idx in range(self.num_ports)]
        )

    def check_capacity_constraints(self, prices: np.ndarray, port_idx: int) -> bool:
        """Check whether a port's combined volume stays within its capacity."""
        self._validate_prices(prices)
        total_volume = sum(
            self.calculate_market_share(prices, port_idx, sl_idx)
            for sl_idx in range(self.num_shipping_lines)
        )
        return total_volume <= self.ports[port_idx].capacity + 1e-9

    def initialize_prices(self, method: str = "random") -> np.ndarray:
        """Create a bounded starting matrix for the sequential updates."""
        if method == "random":
            prices = np.random.uniform(
                self.market_params.min_price,
                self.market_params.max_price,
                (self.num_ports, self.num_shipping_lines),
            )
        elif method == "cost_plus":
            prices = np.array(
                [np.asarray(port.costs, dtype=float) + 30.0 for port in self.ports]
            )
            prices = np.clip(
                prices,
                self.market_params.min_price,
                self.market_params.max_price,
            )
        elif method == "uniform":
            uniform_price = (
                self.market_params.min_price + self.market_params.max_price
            ) / 2
            prices = np.full(
                (self.num_ports, self.num_shipping_lines), uniform_price
            )
        else:
            raise ValueError(f"Unknown initialization method: {method}")

        self._validate_prices(prices)
        if not all(
            self.check_capacity_constraints(prices, port_idx)
            for port_idx in range(self.num_ports)
        ):
            # A feasible upper-bound matrix provides a defined start when a
            # randomly generated matrix violates a shared capacity limit.
            prices = np.full(
                (self.num_ports, self.num_shipping_lines),
                self.market_params.max_price,
            )
            if not all(
                self.check_capacity_constraints(prices, port_idx)
                for port_idx in range(self.num_ports)
            ):
                raise ValueError(
                    "The upper-bound price matrix is infeasible for port capacity."
                )
        return prices

    def best_response_price(
        self,
        prices: np.ndarray,
        port_idx: int,
        shipping_line_idx: int,
    ) -> float:
        """Find the best capacity-feasible price on a finite candidate grid."""
        self._validate_prices(prices)
        cost = self.ports[port_idx].costs[shipping_line_idx]
        candidates = np.linspace(
            self.market_params.min_price,
            self.market_params.max_price,
            100,
        )
        # Including the current price guarantees a feasible option whenever
        # the current matrix is feasible, even if it is not on the grid.
        candidates = np.append(candidates, prices[port_idx, shipping_line_idx])
        best_price = None
        best_profit = -np.inf

        for candidate in candidates:
            test_prices = prices.copy()
            test_prices[port_idx, shipping_line_idx] = candidate
            if not self.check_capacity_constraints(test_prices, port_idx):
                continue

            volume = self.calculate_market_share(
                test_prices, port_idx, shipping_line_idx
            )
            profit = (candidate - cost) * volume
            if profit > best_profit:
                best_profit = profit
                best_price = float(candidate)

        if best_price is None:
            raise RuntimeError(
                "No capacity-feasible price candidate was found."
            )
        return best_price

    def iterate_best_response(
        self,
        prices: np.ndarray,
        convergence_tolerance: float = 0.01,
    ) -> Tuple[np.ndarray, int, Dict]:
        """Apply sequential coordinate updates until stable or at the limit."""
        if convergence_tolerance <= 0:
            raise ValueError("Convergence tolerance must be positive.")

        prices = np.asarray(prices, dtype=float).copy()
        self._validate_prices(prices)
        if not all(
            self.check_capacity_constraints(prices, port_idx)
            for port_idx in range(self.num_ports)
        ):
            raise ValueError("Initial prices must satisfy all capacity limits.")

        max_iterations = 1000
        iteration = 0
        converged = False
        price_change = np.inf
        self.price_history = [prices.copy()]
        self.profit_history = [self.calculate_all_profits(prices)]

        while not converged and iteration < max_iterations:
            iteration += 1
            old_prices = prices.copy()

            for port_idx in range(self.num_ports):
                for sl_idx in range(self.num_shipping_lines):
                    prices[port_idx, sl_idx] = self.best_response_price(
                        prices, port_idx, sl_idx
                    )

            self.price_history.append(prices.copy())
            self.profit_history.append(self.calculate_all_profits(prices))
            price_change = float(np.max(np.abs(prices - old_prices)))
            converged = price_change < convergence_tolerance

        results = {
            "converged": converged,
            "iterations": iteration,
            "final_price_change": price_change,
            "equilibrium_prices": prices.copy(),
            "equilibrium_profits": self.calculate_all_profits(prices),
        }
        return prices, iteration, results

    def analyze_convergence(self) -> Dict:
        """Summarize changes in recorded prices and profits by iteration."""
        if len(self.price_history) < 2:
            return {"error": "Insufficient history for analysis"}

        price_changes = [
            float(np.max(np.abs(self.price_history[i] - self.price_history[i - 1])))
            for i in range(1, len(self.price_history))
        ]
        profit_evolution = np.asarray(self.profit_history)
        if len(price_changes) >= 10:
            recent_changes = price_changes[-10:]
            oscillation_detected = (
                np.std(recent_changes) > np.mean(recent_changes)
            )
        else:
            oscillation_detected = False

        return {
            "price_changes": price_changes,
            "profit_evolution": profit_evolution,
            "convergence_rate": float(np.mean(price_changes)),
            "oscillation_detected": oscillation_detected,
            "final_stability": price_changes[-1],
        }

In [ ]:
def create_concrete_example():
    """Build the three-port, three-shipping-line example."""
    ports = [
        Port(id=1, name="Port A", capacity=20000, costs=[80, 85, 82]),
        Port(id=2, name="Port B", capacity=18000, costs=[85, 78, 88]),
        Port(id=3, name="Port C", capacity=22000, costs=[75, 90, 80]),
    ]
    shipping_lines = [
        ShippingLine(id=1, name="Global Shipping Line", demand=12000),
        ShippingLine(id=2, name="Pacific Trade Line", demand=15000),
        ShippingLine(id=3, name="Atlantic Container Line", demand=10000),
    ]
    market_params = MarketParameters(
        price_sensitivity=0.01,
        min_price=80,
        max_price=150,
    )
    return ports, shipping_lines, market_params


ports, shipping_lines, market_params = create_concrete_example()
solver = IterativeBestResponse(ports, shipping_lines, market_params)

print("Problem Configuration:")
print(f"Number of ports: {len(ports)}")
print(f"Number of shipping lines: {len(shipping_lines)}")
print(f"Total market demand: {sum(line.demand for line in shipping_lines):,} TEU")
print(f"Total port capacity: {sum(port.capacity for port in ports):,} TEU")
print(f"Price sensitivity (beta): {market_params.price_sensitivity}")
print(
    f"Price range: ${market_params.min_price} - "
    f"${market_params.max_price} per demand unit"
)

In [ ]:
def test_initialization_strategies(ports, shipping_lines, market_params):
    """Compare outcomes from the three specified starting-price rules."""
    strategies = ["random", "cost_plus", "uniform"]
    results = {}

    for strategy in strategies:
        print(f"\nTesting {strategy} initialization:")
        print("-" * 40)
        solver_strategy = IterativeBestResponse(
            ports, shipping_lines, market_params
        )
        initial_prices = solver_strategy.initialize_prices(strategy)
        equilibrium_prices, iterations, result_data = (
            solver_strategy.iterate_best_response(
                initial_prices, convergence_tolerance=0.01
            )
        )

        results[strategy] = {
            "initial_prices": initial_prices,
            "equilibrium_prices": equilibrium_prices,
            "iterations": iterations,
            "converged": result_data["converged"],
            "final_profits": result_data["equilibrium_profits"],
            "solver": solver_strategy,
        }
        print(f"Converged: {result_data['converged']}")
        print(f"Iterations: {iterations}")
        print(f"Final maximum price change: ${result_data['final_price_change']:.4f}")
        print(
            "Total system profit: "
            f"${np.sum(result_data['equilibrium_profits']):,.2f}"
        )

        price_df = pd.DataFrame(
            equilibrium_prices,
            index=[port.name for port in ports],
            columns=[line.name for line in shipping_lines],
        )
        print("\nEquilibrium Prices:")
        print(price_df.round(2))

    return results


strategy_results = test_initialization_strategies(
    ports, shipping_lines, market_params
)

In [ ]:
def analyze_convergence_behavior(solver, ports, shipping_lines, title_suffix=""):
    """Print and plot the price and profit history from one solver run."""
    convergence_analysis = solver.analyze_convergence()
    if "error" in convergence_analysis:
        print("Insufficient history for convergence analysis.")
        return convergence_analysis

    print(f"\nConvergence Analysis {title_suffix}:")
    print("=" * 50)
    print(f"Mean maximum price change: {convergence_analysis['convergence_rate']:.6f}")
    print(f"Oscillation diagnostic: {convergence_analysis['oscillation_detected']}")
    print(f"Final maximum price change: ${convergence_analysis['final_stability']:.6f}")

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(
        f"Convergence Analysis {title_suffix}", fontsize=16, fontweight="bold"
    )

    ax1 = axes[0, 0]
    iterations = range(1, len(convergence_analysis["price_changes"]) + 1)
    ax1.plot(iterations, convergence_analysis["price_changes"], "b-", linewidth=2)
    ax1.set_title("Maximum Price Change per Iteration")
    ax1.set_xlabel("Iteration")
    ax1.set_ylabel("Price change (price units)")
    ax1.grid(True, alpha=0.3)

    ax2 = axes[0, 1]
    profit_evolution = convergence_analysis["profit_evolution"]
    iterations_total = range(len(profit_evolution))
    for port_idx, port in enumerate(ports):
        ax2.plot(
            iterations_total,
            profit_evolution[:, port_idx],
            label=port.name,
            linewidth=2,
            marker="o",
            markersize=3,
        )
    ax2.set_title("Profit Evolution by Port")
    ax2.set_xlabel("Recorded iteration")
    ax2.set_ylabel("Profit (price units × TEU)")
    ax2.legend(title="Port")
    ax2.grid(True, alpha=0.3)

    ax3 = axes[1, 0]
    total_profits = np.sum(profit_evolution, axis=1)
    ax3.plot(
        iterations_total,
        total_profits,
        "g-",
        linewidth=2,
        marker="s",
        markersize=3,
    )
    ax3.set_title("Total System Profit Evolution")
    ax3.set_xlabel("Recorded iteration")
    ax3.set_ylabel("Total profit (price units × TEU)")
    ax3.grid(True, alpha=0.3)

    ax4 = axes[1, 1]
    price_matrix = np.asarray(solver.price_history)
    num_combinations = len(ports) * len(shipping_lines)
    heatmap_data = price_matrix.reshape(len(price_matrix), num_combinations).T
    image = ax4.imshow(
        heatmap_data,
        aspect="auto",
        cmap="viridis",
        interpolation="nearest",
    )
    ax4.set_title("Price Evolution Heatmap")
    ax4.set_xlabel("Recorded iteration")
    ax4.set_ylabel("Port–shipping-line pair")
    labels = [
        f"{port.name} — {line.name}"
        for port in ports
        for line in shipping_lines
    ]
    ax4.set_yticks(range(num_combinations))
    ax4.set_yticklabels(labels, fontsize=8)
    fig.colorbar(image, ax=ax4, label="Price (price units)")

    plt.tight_layout()
    plt.show()
    return convergence_analysis


best_solver = strategy_results["random"]["solver"]
convergence_data = analyze_convergence_behavior(
    best_solver,
    ports,
    shipping_lines,
    "(Random Initialization)",
)

In [ ]:
def compare_equilibrium_quality(strategy_results, ports, shipping_lines):
    """Compare profits, price dispersion, concentration, and iteration counts."""
    comparison_data = []

    for strategy, data in strategy_results.items():
        equilibrium_prices = data["equilibrium_prices"]
        final_profits = data["final_profits"]
        total_profit = float(np.sum(final_profits))
        profit_variance = float(np.var(final_profits))
        price_variance = float(np.var(equilibrium_prices))

        volumes = [
            data["solver"].calculate_market_share(
                equilibrium_prices, port_idx, sl_idx
            )
            for port_idx in range(len(ports))
            for sl_idx in range(len(shipping_lines))
        ]
        total_demand = sum(line.demand for line in shipping_lines)
        volume_shares = [volume / total_demand for volume in volumes]
        hhi = float(sum(share**2 for share in volume_shares))

        comparison_data.append(
            {
                "Strategy": strategy,
                "Total Profit": total_profit,
                "Profit Variance": profit_variance,
                "Price Variance": price_variance,
                "Cell-level HHI": hhi,
                "Iterations": data["iterations"],
                "Converged": data["converged"],
            }
        )

    comparison_df = pd.DataFrame(comparison_data)
    print("\nEquilibrium Quality Comparison:")
    print("=" * 80)
    print(comparison_df.round(4))

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(
        "Outcomes Across Initialization Strategies",
        fontsize=16,
        fontweight="bold",
    )

    plot_specs = [
        ("Total Profit", "Total profit (price units × TEU)", "Total System Profit"),
        ("Iterations", "Iterations", "Iterations to Stopping Criterion"),
        ("Cell-level HHI", "Cell-level HHI", "Volume Concentration Across Port–Line Cells"),
        ("Price Variance", "Price variance (price units²)", "Price Variance"),
    ]
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]
    for ax, (column, ylabel, title) in zip(axes.flat, plot_specs):
        bars = ax.bar(
            comparison_df["Strategy"],
            comparison_df[column],
            color=colors[: len(comparison_df)],
        )
        ax.set_title(title)
        ax.set_ylabel(ylabel)
        ax.tick_params(axis="x", rotation=30)
        for bar, value in zip(bars, comparison_df[column]):
            ax.annotate(
                f"{value:,.3g}",
                (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3),
                textcoords="offset points",
                ha="center",
                va="bottom",
            )

    plt.tight_layout()
    plt.show()
    return comparison_df


quality_comparison = compare_equilibrium_quality(
    strategy_results, ports, shipping_lines
)

In [ ]:
def tolerance_sensitivity_analysis(ports, shipping_lines, market_params):
    """Study how the price-change stopping tolerance affects each run."""
    tolerances = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1]
    tolerance_results = []

    for tolerance in tolerances:
        solver_tol = IterativeBestResponse(ports, shipping_lines, market_params)
        initial_prices = solver_tol.initialize_prices("random")
        _, iterations, result_data = solver_tol.iterate_best_response(
            initial_prices, convergence_tolerance=tolerance
        )
        tolerance_results.append(
            {
                "Tolerance": tolerance,
                "Iterations": iterations,
                "Total Profit": np.sum(result_data["equilibrium_profits"]),
                "Final Price Change": result_data["final_price_change"],
                "Converged": result_data["converged"],
            }
        )

    tolerance_df = pd.DataFrame(tolerance_results)
    print("\nTolerance Sensitivity Analysis:")
    print("=" * 60)
    print(tolerance_df.round(4))

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(
        "Convergence Tolerance Sensitivity Analysis",
        fontsize=16,
        fontweight="bold",
    )

    axes[0, 0].plot(
        tolerance_df["Tolerance"],
        tolerance_df["Iterations"],
        "bo-",
        linewidth=2,
        markersize=8,
    )
    axes[0, 0].set_title("Iterations vs. Tolerance")
    axes[0, 0].set_xlabel("Convergence tolerance (price units)")
    axes[0, 0].set_ylabel("Iterations")
    axes[0, 0].set_xscale("log")
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(
        tolerance_df["Tolerance"],
        tolerance_df["Total Profit"],
        "ro-",
        linewidth=2,
        markersize=8,
    )
    axes[0, 1].set_title("Total Profit vs. Tolerance")
    axes[0, 1].set_xlabel("Convergence tolerance (price units)")
    axes[0, 1].set_ylabel("Total profit (price units × TEU)")
    axes[0, 1].set_xscale("log")
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(
        tolerance_df["Tolerance"],
        tolerance_df["Final Price Change"],
        "go-",
        linewidth=2,
        markersize=8,
    )
    axes[1, 0].set_title("Final Price Change vs. Tolerance")
    axes[1, 0].set_xlabel("Convergence tolerance (price units)")
    axes[1, 0].set_ylabel("Final maximum price change (price units)")
    axes[1, 0].set_xscale("log")
    axes[1, 0].grid(True, alpha=0.3)

    efficiency = tolerance_df["Total Profit"] / tolerance_df["Iterations"]
    axes[1, 1].plot(
        tolerance_df["Tolerance"],
        efficiency,
        "mo-",
        linewidth=2,
        markersize=8,
    )
    axes[1, 1].set_title("Profit per Iteration vs. Tolerance")
    axes[1, 1].set_xlabel("Convergence tolerance (price units)")
    axes[1, 1].set_ylabel("Profit per iteration")
    axes[1, 1].set_xscale("log")
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return tolerance_df


tolerance_results = tolerance_sensitivity_analysis(
    ports, shipping_lines, market_params
)

In [ ]:
def scalability_analysis(market_params):
    """Measure run time and outcomes on several generated problem sizes."""
    import time

    test_cases = [
        {"num_ports": 2, "num_shipping_lines": 2, "name": "2 Ports, 2 Lines"},
        {"num_ports": 3, "num_shipping_lines": 2, "name": "3 Ports, 2 Lines"},
        {"num_ports": 3, "num_shipping_lines": 3, "name": "3 Ports, 3 Lines"},
        {"num_ports": 4, "num_shipping_lines": 3, "name": "4 Ports, 3 Lines"},
        {"num_ports": 5, "num_shipping_lines": 4, "name": "5 Ports, 4 Lines"},
    ]
    scalability_results = []

    for test_case in test_cases:
        num_ports = test_case["num_ports"]
        num_shipping_lines = test_case["num_shipping_lines"]
        test_ports = []
        for i in range(num_ports):
            costs = [80 + i * 5 + j * 3 for j in range(num_shipping_lines)]
            capacity = 15000 + i * 2000
            test_ports.append(
                Port(
                    id=i + 1,
                    name=f"Port {chr(65 + i)}",
                    capacity=capacity,
                    costs=costs,
                )
            )

        test_shipping_lines = [
            ShippingLine(
                id=j + 1,
                name=f"Line {j + 1}",
                demand=10000 + j * 2000,
            )
            for j in range(num_shipping_lines)
        ]
        solver_scale = IterativeBestResponse(
            test_ports, test_shipping_lines, market_params
        )
        initial_prices = solver_scale.initialize_prices("random")

        start_time = time.perf_counter()
        _, iterations, result_data = solver_scale.iterate_best_response(
            initial_prices, convergence_tolerance=0.01
        )
        execution_time = time.perf_counter() - start_time

        scalability_results.append(
            {
                "Problem Size": test_case["name"],
                "Ports": num_ports,
                "Shipping Lines": num_shipping_lines,
                "Price Variables": num_ports * num_shipping_lines,
                "Iterations": iterations,
                "Execution Time (s)": execution_time,
                "Total Profit": np.sum(result_data["equilibrium_profits"]),
                "Converged": result_data["converged"],
            }
        )

    scalability_df = pd.DataFrame(scalability_results)
    print("\nScalability Analysis:")
    print("=" * 80)
    print(scalability_df.round(4))

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(
        "Scalability Analysis — Performance by Problem Size",
        fontsize=16,
        fontweight="bold",
    )
    plot_specs = [
        ("Iterations", "Iterations", "Iterations vs. Price Variables"),
        ("Execution Time (s)", "Execution time (seconds)", "Run Time vs. Price Variables"),
        ("Total Profit", "Total profit (price units × TEU)", "Total Profit vs. Price Variables"),
    ]
    for ax, (column, ylabel, title) in zip(axes.flat[:3], plot_specs):
        ax.plot(
            scalability_df["Price Variables"],
            scalability_df[column],
            "o-",
            linewidth=2,
            markersize=8,
        )
        ax.set_title(title)
        ax.set_xlabel("Number of price variables")
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3)

    ax4 = axes[1, 1]
    efficiency = (
        scalability_df["Total Profit"] / scalability_df["Execution Time (s)"]
    )
    ax4.plot(
        scalability_df["Price Variables"],
        efficiency,
        "mo-",
        linewidth=2,
        markersize=8,
    )
    ax4.set_title("Profit per Second (Descriptive Metric)")
    ax4.set_xlabel("Number of price variables")
    ax4.set_ylabel("Profit per second")
    ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return scalability_df


scalability_results = scalability_analysis(market_params)

## Interpreting the results and choosing this tier

### What the analyses show

- **Initialization comparison:** Comparing starting rules shows whether the sequential updates reach similar outcomes from the tested initial price matrices. It does not establish that all starting points lead to the same equilibrium.
- **Convergence diagnostics:** The maximum price change records the largest coordinate change between full iterations. The oscillation indicator is a simple descriptive diagnostic based on recent changes; it is not a formal convergence test.
- **Equilibrium-quality metrics:** Total profit is the sum of the ports' profits, not a system-wide objective optimized by this algorithm. Profit variance and price variance describe dispersion. The reported HHI is calculated across port–shipping-line cells, so it is not the conventional port-level market-concentration index.
- **Tolerance sensitivity:** A smaller tolerance asks for a smaller change between iterations before stopping. The resulting iteration count and final prices depend on the data and the update path.
- **Scalability:** Run-time measurements are empirical observations for the generated instances and the current computing environment. They should not be interpreted as a general complexity guarantee or as evidence of a particular scaling law.

If a price lies at a bound, that can indicate that the best feasible candidate in the searched grid is at that bound. If a port's capacity is active, its feasible candidate set is restricted by the combined volumes across shipping lines. Multiple price matrices may satisfy the stopping criterion, and the algorithm does not test uniqueness.

### Relationship to Tier 1 and scope of comparison

This tier uses sequential best responses rather than a system-wide mixed-integer optimization objective. Its output is therefore a candidate competitive outcome under the stated choice, cost, capacity, and price-grid assumptions. A numerical comparison with a Tier 1 result requires the same input data, compatible objectives, and a clearly specified comparison criterion; no Tier 1 solution is computed in this notebook.

The method is useful for studying how ports adjust prices against current competitor prices and for examining sensitivity to initialization and stopping tolerance. Its convergence is not guaranteed for every game, and a converged grid-based result is not an exact continuous-price Nash equilibrium certificate. Natural extensions include testing additional demand and cost scenarios or adding market features while stating how they alter the model; those extensions are outside the computations performed here.